# Real `auto-gptq` / `autoawq` on distilgpt2 — vs QA packet quantizers

Runs **real** `auto-gptq` + `autoawq` library quantization on CUDA, alongside
the QA packet quantizers from `experiments/qa_ml/72_hf_tiny_lm_qa_packet_quant.py`,
on identical calibration and eval prompts. Compares head-to-head per bit width.

## How to run

1. **Runtime → Change runtime type → T4 GPU** (the free tier is enough).
2. **Runtime → Run all**.
3. After completion the last cell triggers a browser download of
   `results_real_auto_gptq_awq_colab.json`. Place that file at
   `experiments/qa_ml/results_real_auto_gptq_awq_colab.json` in the repo
   and commit.

## Expected partial failures (handled — the run continues either way)

- **`autoawq` on GPT-2**: AWQ was designed for Llama-family `nn.Linear`
  projections; GPT-2's `Conv1D` may not be supported. We try and record
  whatever happens.
- **`auto-gptq` at 2/3-bit on a 82M-param model**: kernels exist but loss
  can blow up — that's still real-library evidence, just not a good result.

If a method errors, the row in the JSON contains an `"error"` field instead
of `eval_loss`.


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

%pip install -q \
    'transformers==4.45.2' \
    'tokenizers>=0.20,<0.21' \
    'peft==0.7.1' \
    'huggingface_hub' \
    'safetensors'
%pip install -q 'auto-gptq==0.7.1' --no-build-isolation
%pip install -q 'autoawq==0.2.9'

import torch
assert torch.cuda.is_available(), 'Need a GPU runtime (Runtime > Change runtime type > GPU)'
print('torch', torch.__version__, '/ CUDA', torch.version.cuda, '/ device', torch.cuda.get_device_name(0))


In [ ]:
# Pull the QA packet-quant module from the public repo.
!wget -q -O qa_pq_module.py https://raw.githubusercontent.com/1r0nw1ll/quantum-arithmetic-research/main/experiments/qa_ml/72_hf_tiny_lm_qa_packet_quant.py

import importlib.util, sys
spec = importlib.util.spec_from_file_location('qa_pq', 'qa_pq_module.py')
qa_pq = importlib.util.module_from_spec(spec)
sys.modules['qa_pq'] = qa_pq
spec.loader.exec_module(qa_pq)
print('Loaded QA quantizers. BITS =', qa_pq.BITS)
print('CALIBRATION_TEXTS:', len(qa_pq.CALIBRATION_TEXTS), '  EVAL_TEXTS:', len(qa_pq.EVAL_TEXTS))


In [ ]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'distilgpt2'
device = 'cuda'
qa_pq.set_seeds()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

fp_model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
evaluation_gpu  = [tokenizer(t, return_tensors='pt').to(device) for t in qa_pq.EVAL_TEXTS]
calibration_cpu = [tokenizer(t, return_tensors='pt') for t in qa_pq.CALIBRATION_TEXTS]

def causal_loss_and_logits(model, batches):
    model.eval()
    losses, logits_list = [], []
    with torch.no_grad():
        for b in batches:
            out = model(**b, labels=b['input_ids'])
            losses.append(float(out.loss))
            logits_list.append(out.logits.detach().cpu())
    return float(np.mean(losses)), logits_list

def logit_mse(a_list, b_list):
    return float(np.mean([float(torch.mean((a - b) * (a - b))) for a, b in zip(a_list, b_list)]))

fp_loss, fp_logits = causal_loss_and_logits(fp_model, evaluation_gpu)
print(f'FP32 eval loss: {fp_loss:.6f}')


In [ ]:
import copy
rows = []

# QA quantizers + GPTQ/AWQ-shaped proxies — run on CPU model copy (matches
# upstream script) then push the quantized model to GPU for eval.
fp_model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID)
importance = qa_pq.collect_param_importance(fp_model_cpu, calibration_cpu)

for bits in qa_pq.BITS:
    for method in ('gptq_proxy_weighted_clip', 'awq_proxy_scaled_minmax',
                   'qa_lloyd_packet', 'qa_unweighted_lloyd_packet',
                   'qa_affine_residual_packet', 'qa_symmetric_packet'):
        q_model, meta = qa_pq.apply_quantization(fp_model_cpu, bits, method, importance)
        q_model = q_model.to(device)
        q_loss, q_logits = causal_loss_and_logits(q_model, evaluation_gpu)
        meta['eval_loss'] = q_loss
        meta['loss_delta_vs_fp32'] = q_loss - fp_loss
        meta['logit_mse_vs_fp32'] = logit_mse(fp_logits, q_logits)
        rows.append(meta)
        del q_model; torch.cuda.empty_cache()
    q_model, meta = qa_pq.apply_qa_calibrated_selector(fp_model_cpu, bits, importance, calibration_cpu)
    q_model = q_model.to(device)
    q_loss, q_logits = causal_loss_and_logits(q_model, evaluation_gpu)
    meta['eval_loss'] = q_loss
    meta['loss_delta_vs_fp32'] = q_loss - fp_loss
    meta['logit_mse_vs_fp32'] = logit_mse(fp_logits, q_logits)
    rows.append(meta)
    del q_model; torch.cuda.empty_cache()

print(f'QA + proxy rows: {len(rows)}')
for r in rows:
    print(f"  {r['bits']}-bit {r['method']:>36}: loss={r['eval_loss']:.4f}  logit_mse={r['logit_mse_vs_fp32']:.4f}")


In [ ]:
import os, shutil
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

for bits in qa_pq.BITS:
    print(f'--- REAL auto-gptq @ {bits}-bit ---')
    work = f'/tmp/ag_w{bits}'
    shutil.rmtree(work, ignore_errors=True); os.makedirs(work)
    try:
        qc = BaseQuantizeConfig(bits=bits, group_size=128, desc_act=False)
        ag = AutoGPTQForCausalLM.from_pretrained(MODEL_ID, qc)
        ag.quantize(calibration_cpu, batch_size=1)
        ag.save_quantized(work, use_safetensors=True)
        del ag; torch.cuda.empty_cache()
        ag = AutoGPTQForCausalLM.from_quantized(work, device='cuda:0')
        ag_loss, ag_logits = causal_loss_and_logits(ag, evaluation_gpu)
        rows.append({
            'method': 'real_auto_gptq',
            'library': 'auto-gptq==0.7.1',
            'bits': bits, 'group_size': 128, 'desc_act': False,
            'eval_loss': ag_loss,
            'loss_delta_vs_fp32': ag_loss - fp_loss,
            'logit_mse_vs_fp32': logit_mse(fp_logits, ag_logits),
        })
        print(f'  loss={ag_loss:.4f}  Δ={ag_loss-fp_loss:+.4f}  logit_mse={logit_mse(fp_logits, ag_logits):.4f}')
        del ag; torch.cuda.empty_cache()
    except Exception as e:
        rows.append({'method': 'real_auto_gptq', 'bits': bits, 'error': repr(e)[:500]})
        print(f'  ERROR: {e!r}')


In [ ]:
# autoawq: try 4-bit (its native setting). AWQ was designed for Llama-family
# Linear projections; GPT-2's Conv1D may not be supported. Honest fail is fine.
try:
    from awq import AutoAWQForCausalLM
    print('--- REAL autoawq @ 4-bit ---')
    work = '/tmp/aw_w4'
    shutil.rmtree(work, ignore_errors=True); os.makedirs(work)
    qc = {'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}
    aw = AutoAWQForCausalLM.from_pretrained(MODEL_ID, low_cpu_mem_usage=True, use_cache=False)
    aw.quantize(tokenizer, quant_config=qc, calib_data=qa_pq.CALIBRATION_TEXTS)
    aw.save_quantized(work)
    del aw; torch.cuda.empty_cache()
    aw = AutoAWQForCausalLM.from_quantized(work, fuse_layers=False, device_map='cuda:0')
    inner = aw.model if hasattr(aw, 'model') else aw
    aw_loss, aw_logits = causal_loss_and_logits(inner, evaluation_gpu)
    rows.append({
        'method': 'real_autoawq',
        'library': 'autoawq==0.2.9',
        'bits': 4, 'group_size': 128,
        'eval_loss': aw_loss,
        'loss_delta_vs_fp32': aw_loss - fp_loss,
        'logit_mse_vs_fp32': logit_mse(fp_logits, aw_logits),
    })
    print(f'  loss={aw_loss:.4f}  Δ={aw_loss-fp_loss:+.4f}  logit_mse={logit_mse(fp_logits, aw_logits):.4f}')
except Exception as e:
    rows.append({'method': 'real_autoawq', 'bits': 4, 'error': repr(e)[:500]})
    print(f'  ERROR: {e!r}')


In [ ]:
import json, math

summary = {}
for bits in qa_pq.BITS:
    subset = [r for r in rows if r.get('bits') == bits and 'eval_loss' in r]
    qa = [r for r in subset if r['method'].startswith('qa_')]
    real_gptq = [r for r in subset if r['method'] == 'real_auto_gptq']
    real_awq  = [r for r in subset if r['method'] == 'real_autoawq']
    proxy_gptq = [r for r in subset if r['method'] == 'gptq_proxy_weighted_clip']
    proxy_awq  = [r for r in subset if r['method'] == 'awq_proxy_scaled_minmax']
    def pick(rs, key='eval_loss'):
        return min(rs, key=lambda r: r[key]) if rs else None
    summary[str(bits)] = {
        'best_qa_loss': pick(qa),
        'best_qa_logit': pick(qa, 'logit_mse_vs_fp32'),
        'real_auto_gptq': real_gptq[0] if real_gptq else None,
        'real_autoawq':  real_awq[0]  if real_awq  else None,
        'proxy_gptq': proxy_gptq[0] if proxy_gptq else None,
        'proxy_awq':  proxy_awq[0]  if proxy_awq  else None,
    }

result = {
    'experiment': 'hf_distilgpt2_qa_packet_vs_real_auto_gptq_awq_colab',
    'model_id': MODEL_ID,
    'seed': qa_pq.SEED,
    'bits': qa_pq.BITS,
    'fp32_eval_loss': fp_loss,
    'fp32_eval_ppl': math.exp(fp_loss) if fp_loss < 50 else float('inf'),
    'device': torch.cuda.get_device_name(0),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'claim_boundary': (
        'Real HF distilgpt2; QA packet-quant methods imported from upstream '
        '72_hf_tiny_lm_qa_packet_quant.py and run on GPU; auto-gptq and autoawq '
        'run as real CUDA libraries (not proxies).'
    ),
    'quantization_results': rows,
    'head_to_head_summary': summary,
}

out_path = 'results_real_auto_gptq_awq_colab.json'
with open(out_path, 'w') as f:
    f.write(json.dumps(result, sort_keys=True, separators=(',', ':'), ensure_ascii=False) + '\n')

print('Wrote', out_path)
print('\n=== HEAD-TO-HEAD ===')
for bits, s in summary.items():
    print(f'\n[{bits}-bit]')
    if s['best_qa_loss']:
        print(f"  best QA loss    : {s['best_qa_loss']['method']:>36s}  {s['best_qa_loss']['eval_loss']:.4f}")
    if s['real_auto_gptq']:
        r = s['real_auto_gptq']
        if 'eval_loss' in r:
            print(f"  REAL auto-gptq  : {'auto-gptq':>36s}  {r['eval_loss']:.4f}")
        else:
            print(f"  REAL auto-gptq  : ERROR {r.get('error','?')[:120]}")
    if s['real_autoawq']:
        r = s['real_autoawq']
        if 'eval_loss' in r:
            print(f"  REAL autoawq    : {'autoawq':>36s}  {r['eval_loss']:.4f}")
        else:
            print(f"  REAL autoawq    : ERROR {r.get('error','?')[:120]}")
    if s['proxy_gptq']:
        print(f"  proxy gptq      : {'gptq_proxy_weighted_clip':>36s}  {s['proxy_gptq']['eval_loss']:.4f}")
    if s['proxy_awq']:
        print(f"  proxy awq       : {'awq_proxy_scaled_minmax':>36s}  {s['proxy_awq']['eval_loss']:.4f}")

try:
    from google.colab import files
    files.download(out_path)
except Exception as e:
    print(f'\n(Could not auto-download — fetch {out_path} from the file browser. {e!r})')
